# Nemotron: frozen test evaluation on 67 clusters

**Use a fresh A100 runtime and run top to bottom. No training.**

Upload `frozen-test-inference-v1.zip` to **MyDrive/GTC2027/SCPCP000004/** before running.
The existing V1 adapter must be at `nemotron_4b_expert_lora_v1/adapter/` in that folder.
Do not upload the expert workbook or annotation ledger. The package contains expression evidence only.

This runs the frozen untuned and V1 models on 67 test cases: **134 identity attempts**,
then **134 separate review-only attempts** using the existing review-first prompt.
No V2/V3, new model selection, training, or parameter sweeps.

Every answer is saved immediately to Drive. Terminal failures are retained, never automatically retried.
Each invocation stops after 70 minutes; **rerun Step 5 only to resume**, without reloading the model.
The cap is checked between answers; individual generation limits are checked between tokens.

Earlier A100 timings suggest roughly **2–3 hours of generation**, plus installation/loading;
this is an estimate, not a guarantee or dollar-cost claim. Test cases may take different amounts of time.
Score against expert references locally after returning the ZIP. Test scoring is not performed in Colab.


## 1. Mount Drive and verify the inference-only package


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib,json,sys,zipfile
from pathlib import Path
import os
ROOT = Path(os.environ.get('GTC_DRIVE_ROOT', '/content/drive/MyDrive/GTC2027/SCPCP000004'))
PACKAGE=Path('/content/frozen-test-inference-v1')
ADAPTER_DIR=ROOT/'nemotron_4b_expert_lora_v1/adapter'
package_zip=ROOT/'frozen-test-inference-v1.zip'
assert package_zip.is_file(), 'Upload frozen-test-inference-v1.zip into '+str(ROOT)
EXPECTED_PACKAGE_SHA256 = ''  # Paste the SHA-256 printed by the data-preparation notebook.
assert len(EXPECTED_PACKAGE_SHA256) == 64, 'Set the package hash from the local data-preparation run.'
assert hashlib.sha256(package_zip.read_bytes()).hexdigest()==EXPECTED_PACKAGE_SHA256, 'Evidence package differs.'
with zipfile.ZipFile(package_zip) as z:
    assert set(z.namelist())=={'config.json','case-evidence.jsonl','matched_evaluation.py','frozen_test_evaluation.py'}
    PACKAGE.mkdir(exist_ok=True)
    for name in z.namelist():(PACKAGE/name).write_bytes(z.read(name))
test_config=json.loads((PACKAGE/'config.json').read_text())
for name,expected in test_config['files'].items():
    assert hashlib.sha256((PACKAGE/name).read_bytes()).hexdigest()==expected,name
for name,expected in test_config['adapter_hashes'].items():
    assert (ADAPTER_DIR/name).is_file(), 'Missing saved V1 adapter file: '+name
    assert hashlib.sha256((ADAPTER_DIR/name).read_bytes()).hexdigest()==expected, 'V1 adapter differs: '+name
sys.path.insert(0,str(PACKAGE))
from frozen_test_evaluation import prepare_test_cases,run_test_evaluation
print('67 expression packets and the original V1 adapter verified. Expert answers are not loaded.')


## 2. Install dependencies before importing GPU extensions


In [ ]:
import importlib,importlib.util,importlib.metadata as metadata
import os,subprocess,sys
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'Select an A100 GPU.'
core=['transformers==4.53.0','peft==0.17.1','lm-format-enforcer==0.11.3',
      'jsonschema==4.26.0','accelerate==1.15.0','einops','ninja','packaging']
subprocess.check_call([sys.executable,'-m','pip','install','-q',*core])
os.environ.setdefault('MAX_JOBS','4')
# Complete installation first. Do not import Mamba inside this loop.
for module,package in [('causal_conv1d','causal-conv1d==1.7.0'),
                       ('mamba_ssm','mamba-ssm==2.3.2.post1')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','--no-build-isolation',package])
importlib.invalidate_caches()
# Import and verify the dependency before its consumer.
import causal_conv1d
from causal_conv1d import cpp_functions
assert callable(cpp_functions.causal_conv1d_fwd_function)
assert callable(cpp_functions.causal_conv1d_bwd_function)
import mamba_ssm
from mamba_ssm.ops.triton import ssd_combined
assert callable(ssd_combined.causal_conv1d_fwd_function), 'Restart session: Mamba cached an earlier failed import.'
assert callable(ssd_combined.causal_conv1d_bwd_function)
import transformers,peft
assert transformers.__version__=='4.53.0' and peft.__version__=='0.17.1'
print('Dependency load order verified: torch -> causal-conv -> Mamba -> Transformers/PEFT.')

test_software={name:metadata.version(name) for name in test_config['software']}
assert test_software==test_config['software'], (
    'Stop: environment differs from the recorded development setup. Return this version output before running inference. '+str(test_software))
assert torch.cuda.get_device_name(0)==test_config['gpu'], 'Select the same NVIDIA A100-SXM4-80GB GPU.'


## 3. Load one base model and the saved V1 adapter


In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM
from peft import PeftModel
MODEL_ID=test_config['model_id'];REVISION=test_config['revision']
identity_hash=hashlib.sha256(json.dumps({'model':MODEL_ID,'revision':REVISION,'adapter':test_config['adapter_hashes']},sort_keys=True).encode()).hexdigest()
if 'test_model' not in globals():
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=True)
    assert tokenizer.eos_token_id==11
    base=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,
        trust_remote_code=True,torch_dtype=torch.bfloat16,device_map={'':0})
    test_model=PeftModel.from_pretrained(base,str(ADAPTER_DIR),adapter_name='expert',is_trainable=False)
    del base
    loaded_test_identity=identity_hash
else:
    assert loaded_test_identity==identity_hash, 'Restart before changing model weights.'
test_model.set_adapter('expert');test_model.requires_grad_(False);test_model.eval()
test_model.gradient_checkpointing_disable();test_model.config.use_cache=False
print('One resident base model; V1 adapter is switched off/on for the untuned/adapted comparison.')


## 4. Verify the 67 cases and both frozen prompt formats


In [ ]:
test_cases=prepare_test_cases(PACKAGE,test_config,tokenizer)
for stage,cases in test_cases.items():
    assert len(cases)==67
    limit=test_config['stages'][stage]['decoding']['max_new_tokens']
    assert all(c['input_tokens']+limit<=test_model.config.max_position_embeddings for c in cases)
    print(stage,': 67 cases x 2 model conditions; maximum input tokens:',max(c['input_tokens'] for c in cases))
print('No expert labels, donor identifiers, or sample identifiers are in model prompts.')


## 5. Run / resume test inference
Rerun this cell after its 70-minute session cap. Already-saved attempts, including timeouts,
are skipped. A runtime error stops the run for inspection. Do not delete failed cases or change prompts.
The ZIP is exported even if generation is interrupted. All attempts saved does not mean all answers were valid.


In [ ]:
test_archive=run_test_evaluation(globals(),total_minutes=70)


## 6. Download the output ZIP
Return `nemotron_4b_frozen_test_v1.zip` here. Keep the Drive results folder for resumability.


In [ ]:
from google.colab import files
files.download(str(test_archive))
